# ML-04b : Validité statistique des comparaisons de modèles (Python / sklearn)

**Navigation** : [Version C#](ML-04b-ModelComparison-Validity-CSharp.ipynb) | [Index](README.md) | [<< ML-04](ML-04-Evaluation-Python.ipynb) | [Suivant ML-05 >>](ML-05-TimeSeries-Python.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :

1. **Expliquer** pourquoi un seul score par modèle ne suffit pas à établir qu'un modèle est meilleur qu'un autre.
2. **Construire** un intervalle de confiance bootstrap sur une métrique de régression (R², RMSE) et sur la *différence* entre deux modèles.
3. **Appliquer** le *corrected resampled t-test* (Nadeau & Bengio, 2003) -- une approximation pour comparer deux modèles via la validation croisée k-fold -- et comprendre pourquoi le t-test naïf sur les plis est anti-conservateur.
4. **Quantifier** la taille d'effet et **corriger** les comparaisons multiples (Bonferroni) quand on compare trois modèles ou plus.

## La question centrale

ML-04 a appris à calculer R², RMSE, et à utiliser la validation croisée. Mais considérons deux modèles :
LinearRegression obtient **R² = 0.48**, RandomForest obtient **R² = 0.42**. LinearRegression est-il
*vraiment* le meilleur ?

La réponse honnête est : **on ne peut pas savoir avec un seul nombre**. Le R² mesuré sur un échantillon
est une variable aléatoire -- il varie d'un split à l'autre. Conclure "A > B" exige de savoir si l'écart
observé est **plus grand que le bruit d'échantillonnage**. C'est exactement le rôle d'un test statistique.

Ce notebook est le compagnon méthodologique de ML-04 : là où ML-04 apprend à *calculer* les métriques,
ML-04b apprend à *comparer* les modèles avec des incertitudes et des hypothèses explicites.

## Plan du notebook

Cinq sections + trois exercices, articulés autour de la question « comment savoir si A est *vraiment*
mieux que B ? » :

| # | Section | Question | Méthode |
|---|---------|----------|---------|
| 1 | Variance d'un seul split | Combien de classements différents selon la graine ? | 25 splits avec graines différentes |
| 2 | k-fold moyenne ± écart-type | Suffisant pour conclure ? | KFold n_splits=10 |
| 3 | Bootstrap IC | Quel est l'IC sur la différence A − B ? | Bootstrap 2000 itérations |
| 4 | Corrected resampled t-test | Comment approcher la dépendance des plis ? | Formule Nadeau-Bengio |
| 5 | Taille d'effet + Bonferroni | "Significatif" suffit-il ? Comment comparer 3+ modèles ? | Effet apparié borné, correction α/n |

**Prérequis** : Python, régression et probabilités. **Durée indicative** : 90 minutes.

**Coût de calcul** : 25 splits et 10 plis pour trois modèles, puis 2000 rééchantillonnages sur les prédictions déjà calculées ; le temps dépend du matériel.

**Concepts clés** : variance d'échantillonnage, IC bootstrap, validation croisée k-fold, t-test
apparié, anti-conservatisme des comparaisons naïves, taille d'effet (Cohen's d), comparaisons
multiples (Bonferroni).

**Références** : Nadeau & Bengio 2003 *Inference for the Generalization Error* ; Efron 1979
*Bootstrap methods* ; Cohen 1988 *Statistical power analysis* ; Dietterich 1998 *Approximate
statistical tests for comparing supervised classification learning algorithms*.

In [1]:
# Wiring : modeles de regression sklearn + dataset reel (diabetes, bundled sklearn -- pas de telechargement).
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error

RANDOM_STATE = 42
data = load_diabetes()
X, y = data.data, data.target
print(f"Diabetes : {X.shape[0]} patients, {X.shape[1]} variables (target = progression de la maladie apres 1 an)")

# Trois modeles de capacites croissantes : lineaire (lineaire pur), foret (non-lineaire, bagging),
# boosting (non-lineaire, boosting). Ce sont de vraies familles d'algorithmes, pas des jouets.
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE),
    "GradientBoosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

Diabetes : 442 patients, 10 variables (target = progression de la maladie apres 1 an)


**Lecture du wiring sklearn + diabetes** (cellule ci-dessus) :

La cellule ci-dessus importe les modules sklearn nécessaires et charge le
dataset `diabetes` :
- `sklearn.datasets.load_diabetes()` : 442 patients, 10 variables prédictives
  (age, sex, bmi, bp, et 6 mesures sériques), cible quantitative (progression
  de la maladie sur 1 an). C'est un *dataset régression* standard, plus
  exigeant que `iris` ou `digits`.
- `train_test_split(...)` : split 75/25, avec une graine fixée.
- 3 modèles : `LinearRegression`, `RandomForestRegressor`, `GradientBoostingRegressor`.

**Pourquoi le dataset `diabetes`** : c'est un *benchmark régression* du bundle
sklearn. Il a l'avantage d'être (a) petit (442 observations), (b) multi-variable
(10 features), (c) avec une cible quantitative -- donc adapté à toutes les
métriques (R², RMSE, MAE). C'est l'équivalent régression de MNIST pour la
classification.

**Sortie attendue** : un message console indiquant les dimensions du dataset
(442, 10) ; les modèles sont définis mais pas encore entraînés.

**Coût indicatif** : chargement du dataset et définition des modèles.

## Section 1 -- La variance d'un seul split

Un split train/test donne **un** R² pour chaque modèle. Avec une seule graine, on
ne peut pas savoir si la différence observée est réelle ou due au hasard de
l'échantillonnage. Cette section illustre *empiriquement* la variance : avec 25
graines différentes, combien de classements observes-t-on ?

**Pourquoi cette section est fondamentale** : avant même de parler de bootstrap
ou de t-test, il faut *voir* la variance d'un seul split. Sans cette intuition,
les sections suivantes paraissent abstraites. C'est le « moment eurêka » :
« ah, donc avec une seule graine j'aurais pu me tromper complètement ! ».

**Le protocole** :
1. Charger le dataset `diabetes` (bundle sklearn) — 442 patients, 10 variables
   prédictives, cible quantitative (progression de la maladie).
2. Pour chaque modèle (LinearRegression, RandomForest, GradientBoosting), faire
   un `train_test_split` avec une graine fixée et noter le R².
3. Répéter l'opération avec 25 graines différentes (0, 1, 2, ..., 24).
4. Compter combien de fois chaque modèle « gagne » (R² le plus élevé).

**Sortie attendue** (cellules code[3]–[4]) : le R² de chaque modèle sur la
graine de départ, puis un compteur global des victoires sur les 25 graines
(sur la sortie committée : LinearRegression 23 splits sur 25, RandomForest 2,
GradientBoosting aucun).

**Interprétation attendue** : un classement *fluctue*. Si on avait choisi *une*
graine au hasard, on aurait pu déclarer LinearRegression vainqueur sans établir un gagnant universel. C'est l'illustration empirique de la
variance d'échantillonnage.

**Coût indicatif** : 25 splits × 3 entraînements.

In [2]:
# Un seul split, une seule graine -> un seul verdict, qui semble solide.
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE)
for name, m in models.items():
    m.fit(X_tr, y_tr)
    print(f"{name:20s} R2 = {r2_score(y_te, m.predict(X_te)):.4f}")
print("\nVerdict de ce split : on classe et on declare un gagnant. Est-il fiable ?")

LinearRegression     R2 = 0.4849


RandomForest         R2 = 0.4556
GradientBoosting     R2 = 0.4240

Verdict de ce split : on classe et on declare un gagnant. Est-il fiable ?


In [3]:
# Repetons avec 25 graines differentes et regardons qui gagne a chaque fois.
winners = []
for seed in range(25):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=seed)
    scores = {}
    for name, m in models.items():
        m.fit(Xtr, ytr)
        scores[name] = r2_score(yte, m.predict(Xte))
    winners.append(max(scores, key=scores.get))

tally = pd.Series(winners).value_counts()
print("Gagnant par split (25 graines) :")
print(tally)
print("\nCes classements decrivent ces splits, pas une garantie pour toutes les graines.")

Gagnant par split (25 graines) :
LinearRegression    23
RandomForest         2
Name: count, dtype: int64

Ces classements decrivent ces splits, pas une garantie pour toutes les graines.


**Interpretation.** Sur ce dataset, LinearRegression gagne la majorité des
splits (23 sur 25 dans la sortie ci-dessus), mais pas tous : RandomForest gagne
les 2 autres, et GradientBoosting aucun. Choisir *une* graine au hasard, c'est
donc prendre le risque de tomber sur une des 2 graines où RandomForest l'emporte
-- et de le déclarer vainqueur. Un classement basé sur un seul split n'est pas
une preuve, c'est un **effet d'échantillonnage**.

## Section 2 -- k-fold : moyenne +/- écart-type (la pratique courante, insuffisante)

La validation croisée k-fold amortit la variance du split. On rapporte souvent
`moyenne +/- écart-type` des R² sur les plis. C'est mieux -- mais **insuffisant
pour conclure** : des écarts-types qui se chevauchent ne prouvent ni l'égalité
ni la supériorité.

**Pourquoi moyenne ± écart-type ne suffit pas** :

La pratique usuelle est de lire la sortie k-fold comme : « LinearRegression
R² = 0.465 ± 0.114, RandomForest R² = 0.401 ± 0.135 » (valeurs de la sortie
committée de la cellule ci-dessous). On regarde si les écarts-types se
chevauchent -- et on conclut qu'ils se chevauchent, donc « pas de différence
significative ». C'est **faux** : le chevauchement des écarts-types n'est *ni
nécessaire ni suffisant* pour conclure à l'égalité.

**Le piège classique** : le score R² varie d’un pli à l’autre. La moyenne de ces scores est elle-même une statistique dont la distribution dépend du protocole. La comparer à la *moyenne* d'un autre modèle
exige un *test* sur la *différence* des moyennes -- pas une inspection visuelle
des écart-types.

**Pour comparer** : la section 3 bootstrappe les observations du test pour deux modèles déjà ajustés. La section 4 examine les différences appariées des plis avec une correction approximative de leur dépendance.

**Sortie attendue** (cellule ci-dessous) : pour chaque modèle, la moyenne et
l'écart-type des R² sur les 10 plis. Les écarts-types se chevauchent large
-- les plis réels vont de 0.30 à 0.60 pour LinearRegression et de 0.10 à 0.57
pour RandomForest sur la sortie committée -- ce qui *suggère* -- à tort -- que
les modèles sont équivalents.

**Coût indicatif** : 10 plis × 3 entraînements.

In [4]:
kf = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}
for name, m in models.items():
    # cross_val_score : R2 negatif possible -> on prend la version standard (R2).
    scores = cross_val_score(m, X, y, cv=kf, scoring="r2")
    cv_results[name] = scores
    print(f"{name:20s} R2 = {scores.mean():.4f} +/- {scores.std():.4f}  (plis: {scores.min():.3f} a {scores.max():.3f})")

print("\nLes barres d'erreur (moyenne +/- 1 ecart-type) se chevauchent entre RandomForest et GradientBoosting.")
print("Peut-on conclure qu'ils sont equivalents ? NON -- chevauchement d'ecart-types n'est pas un test.")

LinearRegression     R2 = 0.4649 +/- 0.1136  (plis: 0.299 a 0.602)


RandomForest         R2 = 0.4011 +/- 0.1349  (plis: 0.097 a 0.570)


GradientBoosting     R2 = 0.3634 +/- 0.1719  (plis: 0.037 a 0.562)

Les barres d'erreur (moyenne +/- 1 ecart-type) se chevauchent entre RandomForest et GradientBoosting.
Peut-on conclure qu'ils sont equivalents ? NON -- chevauchement d'ecart-types n'est pas un test.


**Lecture de la validation croisée k-fold** (cellule ci-dessus) :

La cellule ci-dessus implémente une validation croisée k-fold standard :
- `KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)` : 10 plis,
  avec shuffle (sinon l'ordre des données biaiserait les plis), graine fixée.
- Pour chaque pli : split train/validation, entraînement des 3 modèles,
  calcul du R² sur le pli de validation.
- À la fin : `mean(scores) ± std(scores)` pour chaque modèle.

**Pourquoi k=10** : c'est le compromis usuel entre biais et variance. Le choix influe sur les tailles d’entraînement et de validation ainsi que sur le coût ; ses effets sur la variance dépendent des données. La convention sklearn par défaut est k=5 ; ici on utilise k=10.

**Pourquoi `shuffle=True`** : par défaut, KFold *ne mélange pas* les données.
Si les données sont ordonnées (e.g. par date), les plis successifs seraient
*non-aléatoires*. Le shuffle ne rend pas les scores des plis indépendants : leurs ensembles d’entraînement se chevauchent.

**Sortie** (cellule ci-dessus) : un tableau avec, pour chaque modèle, la moyenne
et l'écart-type des R² sur les 10 plis. Sur la sortie committée :
- LinearRegression : 0.465 ± 0.114 (plis de 0.30 à 0.60)
- RandomForest : 0.401 ± 0.135 (plis de 0.10 à 0.57)
- GradientBoosting : 0.363 ± 0.172 (plis de 0.04 à 0.56)

Les écarts-types se chevauchent -- d'où le piège classique d'en conclure
« pas de différence ».

**Coût indicatif** : 10 plis × 3 entraînements.

### Exercice 1 — Quatre modèles et six comparaisons

Ajoutez ExtraTrees sur les mêmes plis, puis appliquez Bonferroni aux six paires. La fonction ci-dessous reçoit des p-values et doit retourner une décision pour chaque paire ; les fixtures changent les gagnants.

Le seuil de Bonferroni (α divisé par le nombre de comparaisons) est présenté en section 5 : l'exercice peut se traiter après elle.

In [5]:
# Exercice 1 — six décisions pour quatre modèles.
def decisions_bonferroni(p_values, alpha=0.05):
    """Retourner les décisions par paire avec le seuil alpha / nombre de paires."""
    # TODO : vérifier que toutes les p-values fournies sont finies et entre 0 et 1.
    return None

for fixture in (
    {("A", "B"): .001, ("A", "C"): .020, ("A", "D"): .006,
     ("B", "C"): .008, ("B", "D"): .040, ("C", "D"): .009},
    {("A", "B"): .5, ("A", "C"): .002, ("A", "D"): .7,
     ("B", "C"): .003, ("B", "D"): .8, ("C", "D"): .004},
):
    answer = decisions_bonferroni(fixture)
    if answer is None:
        print("Exercice 1 a completer")
    else:
        expected = {pair: p < .05 / len(fixture) for pair, p in fixture.items()}
        print("Contrôle exercice 1 :", set(answer) == set(expected)
              and all(type(answer[k]) is bool and answer[k] == expected[k] for k in expected))


Exercice 1 a completer
Exercice 1 a completer


## Section 3 -- Le bootstrap : tester la différence

### Pourquoi le t-test naïf sur les plis demande de la prudence

Le réflexe classique est de faire un t-test apparié sur les k paires de scores
(une par pli). Dix paires ne suffisent pas à garantir les conditions d'un t-test classique. La dépendance peut sous-estimer la variance et conduire à des p-values trop basses.

**Pourquoi être prudent** : les ensembles d'entraînement des plis se chevauchent ; les différences de scores ne sont donc pas indépendantes. Les ensembles de validation eux-mêmes sont disjoints. Un t-test apparié naïf peut sous-estimer l'incertitude.

**Correction de Nadeau et Bengio** : pour les différences $d_i$, leur variance empirique $s_d^2$ et $k$ plis, l'approximation employée en section 4 est

$$\widehat{\operatorname{Var}}(\bar d) = s_d^2\left(\frac{1}{k} + \frac{n_{test}}{n_{train}}\right).$$

Cette correction approximative ne garantit pas un niveau exact pour toute distribution ou tout protocole.

**Sortie attendue** (cellule suivante) : une fonction
`bootstrap_metric_diff(model_a, model_b, X_te, y_te, metric, n_boot=1000, seed=42)`
qui retourne un IC bootstrap sur la différence des métriques. Cet intervalle décrit l’incertitude du test conditionnellement aux modèles ajustés, sous des hypothèses de rééchantillonnage adaptées.

**Coût indicatif** : 2000 rééchantillonnages des prédictions mises en cache, sans nouvel entraînement.

In [6]:
def bootstrap_metric_diff(model_a, model_b, X_te, y_te, metric, n_boot=1000, seed=RANDOM_STATE):
    """IC bootstrap a 95% sur metric(A) - metric(B), calcule sur le jeu de test.

    Retourne (ic_bas, ic_haut) de la DIFFERENCE. Sous des hypotheses adequates, un IC qui exclut 0 suggere un ecart.
    """
    rng = np.random.RandomState(seed)
    n = len(y_te)
    diffs = np.empty(n_boot)
    pa, pb = model_a.predict(X_te), model_b.predict(X_te)
    for b in range(n_boot):
        idx = rng.randint(0, n, size=n)
        ya = y_te[idx]
        diffs[b] = metric(ya, pa[idx]) - metric(ya, pb[idx])
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return diffs, lo, hi

# Entrainons deux modeles sur le meme train set, puis echantillonnons le test set.
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE)
rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE).fit(X_tr, y_tr)
gb = GradientBoostingRegressor(random_state=RANDOM_STATE).fit(X_tr, y_tr)

diffs, lo, hi = bootstrap_metric_diff(rf, gb, X_te, y_te, r2_score, n_boot=2000)
print(f"Diff R2 (RandomForest - GradientBoosting) : {np.mean(diffs):.4f}")
print(f"IC bootstrap 95% : [{lo:.4f}, {hi:.4f}]")
if lo > 0 or hi < 0:
    print("-> L'IC exclut 0 : l'intervalle exclut 0 sous ce protocole de bootstrap.")
else:
    print("-> L'IC contient 0 : on NE PEUT PAS conclure a une difference significative a 5%.")

Diff R2 (RandomForest - GradientBoosting) : 0.0321
IC bootstrap 95% : [-0.0489, 0.1169]
-> L'IC contient 0 : on NE PEUT PAS conclure a une difference significative a 5%.


**Lecture du bootstrap** (cellule ci-dessus) :

La cellule ci-dessus définit `bootstrap_metric_diff(model_a, model_b, X_te,
y_te, metric, n_boot=1000, seed=42)` -- une fonction qui calcule un IC
bootstrap sur la différence d'une métrique entre deux modèles.

**Algorithme** :
1. Pour b = 1 à n_boot :
   - Tirer un bootstrap de l'ensemble de test : `idx = np.random.choice(n_test, size=n_test, replace=True)`.
   - Calculer `metric_a_b = metric(y_te[idx], y_pred_a[idx])`.
   - Calculer `metric_b_b = metric(y_te[idx], y_pred_b[idx])`.
   - Stocker `diff_b = metric_a_b - metric_b_b`.
2. IC = `[percentile(diff, 2.5), percentile(diff, 97.5)]`.

**Pourquoi bootstrapper sur le test set** : le test set est ce qui *mesure* la
capacité de généralisation. Le rééchantillonnage des observations suppose notamment que celles-ci représentent la population et soient appropriées à un bootstrap i.i.d. Cet IC est conditionnel aux deux modèles ajustés ; il ne capture pas la variabilité de leur réentraînement.

**Pourquoi `replace=True`** : c'est la définition du bootstrap -- on tire avec
remise pour simuler la variabilité d'échantillonnage. Sans remise, on obtient un sous-échantillonnage ; le jackknife retire systématiquement des observations et constitue une autre procédure.

**Sortie** (cellule ci-dessus) : un tuple `(diffs, ic_low, ic_high)` -- les écarts rééchantillonnés et les bornes de l’IC. Sur la sortie committée (paire
RandomForest − GradientBoosting, jeu de test) :
- `mean_diff = 0.032` en faveur de RandomForest
- IC 95 % = `[-0.049, 0.117]` : la borne basse est **négative** -- l'IC contient 0,
  on ne peut pas conclure à une différence significative à 5 %

**Coût indicatif** : deux prédictions mises en cache et 2000 rééchantillonnages.

### Exercice 2 : IC bootstrap sur RMSE

L'exercice 2 étudie le RMSE d’un modèle puis la *différence* entre deux modèles. C'est la
question typique : « LinearRegression a-t-il un RMSE significativement plus
petit que RandomForest ? »

**Pourquoi cette généralisation** : comparer deux modèles via la différence de
RMSE est plus informatif que comparer les RMSE séparément. Un IC qui exclut zéro suggère un écart pour ce protocole et sous les hypothèses du bootstrap, sans prouver une supériorité universelle. Une différence non-significative (l'IC contient zéro) laisse la
question ouverte.

**Le protocole** :

1. Charger le dataset `diabetes`.
2. Split train/test (75/25, random_state=42).
3. Entraîner LinearRegression et RandomForest sur le train.
4. Calculez également l’IC du RMSE de LinearRegression seul. Pour b = 1 à 1000 :
   - Tirer un bootstrap de l'ensemble de test (n_test avec remise).
   - Calculer RMSE_A(b) et RMSE_B(b) sur ce bootstrap.
   - Stocker diff(b) = RMSE_A(b) - RMSE_B(b).
5. IC 95% = [percentile(diff, 2.5), percentile(diff, 97.5)].

**Sortie attendue** : un IC à 95% sur la différence RMSE, et une conclusion
binaire (« différence significative » ou « non significative »).

**Pièges** :
- Ne pas oublier que le bootstrap est sur l'ensemble de **test**, pas
  d'entraînement. Réentraîner sur des échantillons bootstrap du jeu d'entraînement mesure une autre composante de l'incertitude : c'est une procédure différente de notre bootstrap sur le test.
- L'IC doit être calculé sur les *différences*, pas sur les R² individuels.
- Si l'IC contient zéro, la différence est *non concluante* -- pas « pas de
  différence ». C'est un point statistique subtil.

**Coût indicatif** : deux prédictions mises en cache et 1000 rééchantillonnages.
La fonction ci-dessous reçoit un fournisseur d’indices, le nombre de tirages et la graine et retourne les deux bornes pour le RMSE seul, puis les deux bornes pour la différence appariée.


In [7]:
# Exercice 2 — IC de RMSE seul et IC de difference appariée.
def intervalles_rmse(y_true, pred_a, pred_b, n_boot, seed, provider):
    """Retourner (lo A, hi A, lo A-B, hi A-B), percentiles 2.5 et 97.5.

    provider(iteration, seed) donne les indices d'un tirage : les utiliser pour
    les deux modeles. Verifier longueurs, indices et valeurs finies.
    """
    # TODO : calculer les RMSE sur tous les tirages, puis les deux IC.
    return None

fixtures = [([0., 2., 5.], [1., 1., 4.], [0., 3., 5.]),
            ([1., 3., 8.], [1., 5., 7.], [3., 3., 8.])]
indices = np.array([[0, 0, 1], [0, 1, 2], [2, 2, 1], [1, 0, 1]])
for seed, (yy, aa, bb) in enumerate(fixtures):
    def provider(iteration, graine):
        return indices[(iteration + graine) % len(indices)]
    draws = np.array([provider(i, seed) for i in range(len(indices))])
    yy, aa, bb = map(np.asarray, (yy, aa, bb))
    ra = np.sqrt(np.mean((yy[draws] - aa[draws]) ** 2, axis=1))
    rb = np.sqrt(np.mean((yy[draws] - bb[draws]) ** 2, axis=1))
    expected = (*np.percentile(ra, [2.5, 97.5]), *np.percentile(ra-rb, [2.5, 97.5]))
    answer = intervalles_rmse(yy, aa, bb, len(indices), seed, provider)
    if answer is None:
        print("Exercice 2 a completer")
    else:
        print("Contrôle exercice 2 :", len(answer) == 4 and np.isfinite(answer).all()
              and np.allclose(answer, expected))
# Sur le jeu de test, construire 1000 lignes avec rng.randint(0, n, size=(1000, n)).


Exercice 2 a completer
Exercice 2 a completer


## Section 4 -- Le corrected resampled t-test (Nadeau & Bengio, 2003)

Le test corrigé de Nadeau-Bengio est une **approximation** pour
comparer deux modèles en validation croisée k-fold. Il corrige la variance
estimée en tenant compte du chevauchement des plis.

**La formule utilisée** :

```python
def corrected_resampled_ttest(scores_a, scores_b, n_test, n_train):
    k = len(scores_a)
    diffs = scores_a - scores_b  # k paires de differences
    mean_diff = np.mean(diffs)
    var_diff = np.var(diffs, ddof=1)  # variance naif sur k plis
    correction = (1.0 / k) + (n_test / n_train)  # facteur de correction
    corrected_var = var_diff * correction
    t_stat = mean_diff / np.sqrt(corrected_var)
    return t_stat, mean_diff, corrected_var
```

**Pourquoi cette correction** : le facteur `(1/k + n_test/n_train)` ajuste la variance pour tenir compte de la dépendance créée par le recouvrement des entraînements. Il s’agit d’une approximation et non d’une garantie exacte.

**Sortie attendue** (cellule suivante) : une fonction `corrected_resampled_ttest`
reprenant cette variance, retournant le t-statistic, les degrés de liberté et la p-value unilatérale.

**Comparaison avec le t-test naïf** : sur la sortie committée de la cellule
suivante (RandomForest contre GradientBoosting, dataset diabetes), le t-test naïf
donne p = 0.0375 (significatif à 5 %). Le t-test corrigé donne p = 0.0993
(non significatif à 5 %). La correction *gonfle* la variance d'un facteur
≈ 2,1, ce qui change la conclusion.

**Coût indicatif** : calcul vectoriel sur k=10 paires.

In [8]:
def corrected_resampled_ttest(scores_a, scores_b, n_test, n_train):
    """Corrected resampled t-test (Nadeau & Bengio, 2003).

    scores_a, scores_b : tableaux des scores par pli des deux modeles (meme CV).
    n_test, n_train : tailles du pli de test / d'entrainement dans la CV.
    Retourne (t_statistique, dl approximes, p_value unilaterale).
    """
    from scipy import stats
    a, b = np.asarray(scores_a, dtype=float), np.asarray(scores_b, dtype=float)
    if (a.ndim != 1 or b.ndim != 1 or len(a) != len(b) or len(a) < 2
            or not np.isfinite(a).all() or not np.isfinite(b).all()
            or not np.isfinite(n_test) or not np.isfinite(n_train)
            or n_test <= 0 or n_train <= 0):
        raise ValueError("scores finis de même taille >= 2 et tailles de plis positives requis")
    k = len(a)
    d = a - b
    d_mean = d.mean()
    d_var = d.var(ddof=1)               # variance empirique des differences
    # Facteur de correction : 1/k + (n_test / n_train)
    correction = (1.0 / k) + (n_test / n_train)
    var_cor = d_var * correction
    t_stat = (d_mean / np.sqrt(var_cor) if var_cor > 0 else
              (0.0 if d_mean == 0 else np.copysign(np.inf, d_mean)))
    df = k - 1
    # Unilateral : H1 "A meilleur que B" (A - B > 0).
    p_value = stats.t.sf(t_stat, df)
    return t_stat, df, p_value

# Fixture numerique independante : 5 differences, ratio 1/4.
probe = corrected_resampled_ttest([.4, .2, 0, .2, .2], np.zeros(5), 1, 4)
probe_reverse = corrected_resampled_ttest(np.zeros(5), [.4, .2, 0, .2, .2], 1, 4)
assert np.allclose(probe, [2.1081851067789197, 4, .051350213747755905])
assert np.isclose(probe_reverse[2], .948649786252244)
assert corrected_resampled_ttest([0, 0], [0, 0], 1, 4)[2] == .5
assert corrected_resampled_ttest([1, 1], [0, 0], 1, 4)[2] == 0
print(f"Fixture independante : t={probe[0]:.6f}, p={probe[2]:.6f}, p inverse={probe_reverse[2]:.6f}")

# Appliquons cette correction approximative a nos 10 plis (RandomForest vs GradientBoosting).
n_total = len(y)
n_test_fold = n_total // 10
n_train_fold = n_total - n_test_fold
t_stat, df, p_val = corrected_resampled_ttest(
    cv_results["RandomForest"], cv_results["GradientBoosting"], n_test_fold, n_train_fold)
print(f"Corrected resampled t-test (RF vs GB) : t = {t_stat:.3f}, dl = {df}, p unilateral = {p_val:.4f}")
if p_val < 0.05:
    print("-> Difference significative (p < 0.05) avec la correction Nadeau-Bengio.")
else:
    print("-> Non significatif (p >= 0.05) avec la correction Nadeau-Bengio.")

# Comparons avec le t-test NAIF (qui ignore la correlation des plis) pour montrer l'optimisme.
from scipy import stats
naive_t, naive_p = stats.ttest_rel(cv_results["RandomForest"], cv_results["GradientBoosting"])
naive_p_one = naive_p / 2 if naive_t > 0 else 1 - naive_p / 2
print(f"\nT-test naif (NON corrige) : t = {naive_t:.3f}, p unilateral = {naive_p_one:.4f}")
print("-> Le t-test naif est plus optimiste ici (p plus petit). C'est le piege.")

Fixture independante : t=2.108185, p=0.051350, p inverse=0.948650
Corrected resampled t-test (RF vs GB) : t = 1.388, dl = 9, p unilateral = 0.0993
-> Non significatif (p >= 0.05) avec la correction Nadeau-Bengio.

T-test naif (NON corrige) : t = 2.013, p unilateral = 0.0375
-> Le t-test naif est plus optimiste ici (p plus petit). C'est le piege.


## Section 5 -- Taille d'effet et comparaisons multiples

« Significatif » ne veut pas dire « important ». Une différence statistiquement
significative peut être *numériquement négligeable* (e.g. R² = 0.48 vs 0.47).
La **taille d'effet** quantifie l'ampleur pratique de la différence, indépendamment
de la taille de l'échantillon.

**Taille d'effet bornée utilisée ici** : pour les différences appariées $d_i$, leur moyenne $\bar d$ et leur variance empirique $s_d^2$, le code calcule $r = \bar d/\sqrt{\bar d^2+s_d^2}$. C'est une transformation bornée de $d_z=\bar d/s_d$ ; ce n'est pas une corrélation rank-biserial. Les seuils usuels de Cohen pour $d$ ne s'appliquent pas directement à $r$.

**Les comparaisons multiples** : quand on compare 3+ modèles (LinearRegression,
RandomForest, GradientBoosting), le risque d'erreur de Type I (« faux positif »)
augmente. Si chaque comparaison a un risque α = 0.05 et qu'on en fait 3, le risque global peut augmenter ; 1 - (1-0.05)³ ≈ 0.14 supposerait des tests indépendants, ce qui ne s’applique pas nécessairement ici. La correction de **Bonferroni** divise
α par le nombre de comparaisons : α' = α/n. Avec n=3, α' = 0.017 -- plus
strict. Il contrôle le risque global d'au moins une fausse découverte à α lorsque les p-values individuelles sont valides, même si les tests sont dépendants.

**Sortie attendue** (cellule suivante) : pour chaque paire de modèles, la
taille d’effet bornée et p-value comparée au seuil Bonferroni.

**Coût indicatif** : calcul sur les scores par pli et correction.

In [9]:
# Transformation bornee du d apparié, ce n’est pas un rank-biserial.
def effect_size_paired(scores_a, scores_b):
    """Taille d'effet r = d_mean / sqrt(d_mean^2 + var(d)) -- interpretable dans [-1, 1]."""
    a, b = np.asarray(scores_a, dtype=float), np.asarray(scores_b, dtype=float)
    if (a.ndim != 1 or b.ndim != 1 or len(a) != len(b) or len(a) < 2
            or not np.isfinite(a).all() or not np.isfinite(b).all()):
        raise ValueError("scores finis de meme taille >= 2 requis")
    d = a - b
    d_mean = d.mean()
    denom = np.sqrt(d_mean**2 + d.var(ddof=1))
    return d_mean / denom if denom > 0 else 0.0

pairs = [
    ("LinearRegression", "RandomForest"),
    ("LinearRegression", "GradientBoosting"),
    ("RandomForest", "GradientBoosting"),
]
n_comparisons = len(pairs)
alpha_bonferroni = 0.05 / n_comparisons
print(f"Seuil Bonferroni pour {n_comparisons} paires : alpha = {alpha_bonferroni:.4f}\n")
for a, b in pairs:
    r_eff = effect_size_paired(cv_results[a], cv_results[b])
    _, _, p_val = corrected_resampled_ttest(cv_results[a], cv_results[b], n_test_fold, n_train_fold)
    sig = "SIGNIFICATIF" if p_val < alpha_bonferroni else "non-significatif"
    print(f"{a:18s} vs {b:18s} : taille d'effet = {r_eff:+.3f}, p = {p_val:.4f} -> {sig}")
print("\nLa taille d'effet distingue 'significatif mais negligeable' (r proche de 0) d'un ecart plus grand, pour ces plis.")

Seuil Bonferroni pour 3 paires : alpha = 0.0167

LinearRegression   vs RandomForest       : taille d'effet = +0.620, p = 0.0594 -> non-significatif
LinearRegression   vs GradientBoosting   : taille d'effet = +0.663, p = 0.0429 -> non-significatif
RandomForest       vs GradientBoosting   : taille d'effet = +0.537, p = 0.0993 -> non-significatif

La taille d'effet distingue 'significatif mais negligeable' (r proche de 0) d'un ecart plus grand, pour ces plis.


### Exercice 3 : stabilité de l'IC bootstrap

L'IC bootstrap dépend du nombre d'itérations `n_boot`. Trop petit, l'IC est
*instable* (varie beaucoup d'une exécution à l'autre). Trop grand, le calcul
ralentit sans gain substantiel.

**La question de la stabilité** : des répétitions indépendantes mesurent l'incertitude Monte-Carlo des bornes ; la précision dépend de la distribution et de la métrique.

**Le protocole** :

1. Pour n_boot ∈ {100, 500, 1000, 5000} :
   - Calculer l'IC bootstrap (comme dans l'exercice 2).
   - Répéter l'opération 10 fois (avec des graines différentes).
   - Mesurer la *largeur moyenne* et l'*écart-type des bornes* sur les 10 runs.
2. Tracer la largeur de l'IC en fonction de n_boot (échelle log).
3. Examiner la stabilité des bornes entre les dix répétitions, sans supposer une largeur décroissante.

**Sortie attendue** : un graphique matplotlib montrant la largeur de l'IC vs
n_boot, et la variabilité observée des bornes sur les dix répétitions.

La largeur de l'IC décrit l'incertitude statistique et ne tend pas nécessairement vers zéro lorsque `n_boot` augmente. C'est l'erreur Monte-Carlo des quantiles estimés qui diminue approximativement en $1/\sqrt{B}$ sous des conditions de régularité.

**Coût indicatif** : 10 répétitions pour chacun des quatre budgets.


In [10]:
# Exercice 3 — variabilite Monte-Carlo des bornes.
def stabilite_bornes(budgets, repetitions, provider):
    """Pour chaque budget, retourner l'ecart-type des bornes basses et hautes.

    provider(budget, seed) retourne (borne_basse, borne_haute) ; appeler les graines
    0..repetitions-1 pour chaque budget. Retourner un dictionnaire budget -> paire.
    """
    # TODO : ne pas conclure que la largeur de l'IC diminue avec le budget.
    return None

for scale in (1., 3.):
    def fixture(budget, seed):
        return scale * seed / budget, 2. + scale * 2 * seed / budget
    budgets = [100, 500, 1000, 5000]
    answer = stabilite_bornes(budgets, 10, fixture)
    expected = {b: (np.std(scale*np.arange(10)/b), np.std(scale*2*np.arange(10)/b)) for b in budgets}
    if answer is None:
        print("Exercice 3 a completer")
    else:
        ok = (set(answer) == set(expected) and all(len(answer[b]) == 2
              and np.isfinite(answer[b]).all() and np.allclose(answer[b], expected[b]) for b in budgets))
        print("Contrôle exercice 3 :", ok)


Exercice 3 a completer
Exercice 3 a completer


## Exercices

Trois exercices pour s'approprier les outils statistiques de comparaison de
modèles. Chaque stub s'exécute **sans erreur** (C.1). Complétez le `TODO`
pour répondre.

**Conventions C.1** : les cellules contiennent des commentaires
`# TODO: a completer` et des corps partiels. Elles s'exécutent de bout en bout
même non-complétées (message d’attente). Python valide la syntaxe mais pas la logique
métier.

**Exercice 1 — Quatre modèles et comparaisons multiples.** Ajoutez ExtraTrees au banc à 10 plis, puis calculez les six comparaisons et leur seuil Bonferroni. Le stub correspondant conserve cette expérience distincte.

**Exercice 2 — IC de RMSE simple et apparié.** Entraînez LinearRegression et RandomForest ; sur les mêmes observations de test, calculez d'abord l'IC du RMSE du premier puis celui de leur différence avec 1000 rééchantillonnages.

**Exercice 3 — Stabilité Monte-Carlo des bornes.** Pour 100, 500, 1000 et 5000 tirages, répétez dix fois avec des graines distinctes ; comparez les bornes et leur variabilité, non une baisse nécessaire de la largeur de l'IC.

**Durée indicative** : 10–15 minutes par exercice ; le temps de calcul dépend du matériel.

Pour l'exercice 1, le quatrième modèle peut être `ExtraTreesRegressor` : utilisez les mêmes plis, puis calculez les trois paires supplémentaires. Le nombre de comparaisons passe de trois à six et le seuil passe de 0,05/3 à 0,05/6. Exposez les paires et décisions, sans modifier les scores déjà obtenus.

Pour l'exercice 2, les mêmes indices doivent servir aux deux modèles à chaque tirage. Les deux tâches sont distinctes : un IC du RMSE de LinearRegression seul, puis un IC de la différence de RMSE avec RandomForest. Un intervalle contenant zéro ne prouve pas l'égalité.


## Conclusion

- **Un classement ou un score isolé ne prouve pas à lui seul une supériorité** : le classement des modèles varie avec la graine dans la section 1.
- **moyenne +/- écart-type** (Section 2) améliore la stabilité mais ne constitue pas un test ; le
  chevauchement des barres d'erreur n'est ni nécessaire ni suffisant pour conclure.
- **Bootstrap** (Section 3) donne un IC conditionnel aux modèles ajustés, sous des hypothèses de rééchantillonnage adaptées.
- **Corrected resampled t-test** (Section 4, Nadeau & Bengio 2003) est une correction approximative de la dépendance des plis ; le t-test naïf peut être anti-conservateur.
- **Taille d'effet + Bonferroni** (Section 5) séparant "significatif" de "important" et contrôlant
  le risque quand on compare plus de deux modèles.

La leçon transversale : comparer des modèles est une **question statistique**, pas seulement une
question de calcul de métrique. C'est le pendant, pour la modélisation prédictive, de ce que la
validité de backtest (PSR, bruit de Sharpe) est pour le trading.

### References

- C. Nadeau and Y. Bengio, *Inference for the Generalization Error*, Machine Learning, 2003.
- T. G. Dietterich, *Approximate Statistical Tests for Comparing Supervised Classification Learning Algorithms*, Neural Computation, 1998.

*Voir aussi : [ML-04](ML-04-Evaluation-Python.ipynb) pour le calcul des métriques de base.*

## À retenir

**Trois concepts à retenir** :

1. **La variance d'échantillonnage compte** : un R² est une mesure sur un échantillon et varie avec les données. Avec 25 graines, on voit le classement *fluctuer*.
2. **Les tests corrigés existent pour une raison** : le t-test naïf peut sous-estimer la variance. Le corrected resampled t-test (Nadeau & Bengio 2003) est une correction approximative.
3. **« Significatif » n'est pas « important »** : la taille d'effet
   (transformation bornée du d apparié) sépare la significativité statistique de
   l'ampleur pratique. Et les comparaisons multiples (Bonferroni) corrigent
   le risque global quand on compare 3+ modèles.

**Pourquoi ce notebook est précieux pour la modélisation ML** : la tentation est
grande de déclarer un modèle « gagnant » sur la base d'une métrique unique.
Ce notebook montre *empiriquement* pourquoi cette tentation est *dangereuse* :
un seul R² est une variable aléatoire, et il faut des outils statistiques
pour conclure.

**Au-delà de la classification/régression** : les mêmes principes
s'appliquent au trading algorithmique (cf. les notebooks QuantConnect sur
la robustesse et le bruit de Sharpe) et à toute comparaison de systèmes
complexes. La leçon transversale : *une mesure isolée ne prouve rien*.

**Coût global indicatif** : 25 splits et 10 plis pour trois modèles, puis 2000 rééchantillonnages ; la durée dépend du matériel.

## Vérifiez votre compréhension

1. Pourquoi les scores de plis peuvent-ils être dépendants ?
2. Quelle incertitude le bootstrap du seul test ne mesure-t-il pas ?
3. Qu'ajuste Bonferroni pour les trois comparaisons ?

<details><summary>Réponses</summary>

1. Les ensembles d'entraînement se chevauchent. 2. La variabilité du réentraînement des modèles. 3. Le seuil de chaque test, de 0,05 à 0,05/3.

</details>

## Pour aller plus loin

- Nadeau et Bengio (2003), *Inference for the Generalization Error*.
- Efron (1979), *Bootstrap methods*.
- [ML-04 — Évaluation](ML-04-Evaluation-Python.ipynb).